In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
!pip install -q transformers==4.52.4 datasets==3.6.0 evaluate jiwer accelerate librosa soundfile sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.7 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.


In [ ]:
import os
os._exit(0)

In [2]:
import transformers
import datasets
import torch

print(transformers.__version__)
print(datasets.__version__)
print(torch.__version__)

4.52.4
3.6.0
2.10.0+cu128


In [3]:
import random
import numpy as np
import torch

from datasets import load_dataset, Audio

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)

import evaluate

2026-06-26 05:42:51.549341: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782452571.729727     112 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782452571.783373     112 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782452572.233465     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782452572.233503     112 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782452572.233506     112 computation_placer.cc:177] computation placer alr

In [5]:
import transformers
import datasets

print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("Torch        :", torch.__version__)

Transformers : 4.52.4
Datasets     : 3.6.0
Torch        : 2.10.0+cu128


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [7]:
seed = 42

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [8]:
train_dataset = load_dataset(
    "librispeech_asr",
    "clean",
    split="train.100[:10%]"
)

test_dataset = load_dataset(
    "librispeech_asr",
    "clean",
    split="test[:10%]"
)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

clean/test/0000.parquet:   0%|          | 0.00/350M [00:00<?, ?B/s]

clean/train.100/0000.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

...

Generating train.100 split:   0%|          | 0/28539 [00:00<?, ? examples/s]

Generating train.360 split:   0%|          | 0/104014 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2703 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

In [9]:
print(len(train_dataset))
print(len(test_dataset))

2854
262


In [10]:
train_dataset = train_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

test_dataset = test_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

In [11]:
sample = train_dataset[0]

print(sample.keys())

dict_keys(['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'])


In [12]:
processor = Wav2Vec2Processor.from_pretrained(
    "facebook/wav2vec2-base-960h"
)

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [13]:
import evaluate

wer_metric = evaluate.load("wer")

In [14]:
def prepare_dataset(batch):

    audio = batch["audio"]

    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch

In [15]:
train_dataset = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names,
    num_proc=1
)

Map:   0%|          | 0/2854 [00:00<?, ? examples/s]

In [16]:
test_dataset = test_dataset.map(
    prepare_dataset,
    remove_columns=test_dataset.column_names,
    num_proc=1
)

Map:   0%|          | 0/262 [00:00<?, ? examples/s]

In [20]:
print(len(train_dataset))
print(len(test_dataset))

print(train_dataset.column_names)

print(train_dataset[0].keys())

print(processor.decode(train_dataset[0]["labels"]))

2854
262
['input_values', 'labels']
dict_keys(['input_values', 'labels'])
CHAPTER SIXTEN I MIGHT HAVE TOLD YOU OF THE BEGINING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SE EVERY STEP BY WHICH WE CAME I TO AGRE TO WHATEVER MARGUERITE WISHED


In [21]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

import torch
from transformers import Wav2Vec2Processor


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    max_length_labels: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    pad_to_multiple_of_labels: Optional[int] = None

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):

        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        with self.processor.as_target_processor():

            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                max_length=self.max_length_labels,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        batch["labels"] = labels

        return batch

In [22]:
data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

In [23]:
batch = data_collator([
    train_dataset[0],
    train_dataset[1]
])

print(batch.keys())

print(batch["input_values"].shape)

print(batch["labels"].shape)

dict_keys(['input_values', 'labels'])
torch.Size([2, 257360])
torch.Size([2, 214])


/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


In [24]:
print(processor.feature_extractor.return_attention_mask)

False


In [25]:
batch = processor.pad(
    [{"input_values": train_dataset[0]["input_values"]}],
    return_tensors="pt",
    padding=True
)

print(batch.keys())

dict_keys(['input_values'])


In [26]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-base-960h"
)

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
batch = data_collator([
    train_dataset[0],
    train_dataset[1]
])

batch = {k: v.to(device) for k, v in batch.items()}

In [32]:
outputs = model(**batch)

print("Loss =", outputs.loss)
print("Is NaN =", torch.isnan(outputs.loss))
print("Logits Shape =", outputs.logits.shape)

Loss = tensor(80.2604, device='cuda:0', grad_fn=<SumBackward0>)
Is NaN = tensor(False, device='cuda:0')
Logits Shape = torch.Size([2, 804, 32])


In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

batch = {k: v.to(device) for k, v in batch.items()}

In [31]:
print(next(model.parameters()).device)
print(batch["input_values"].device)

cuda:0
cuda:0


In [33]:
import numpy as np
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)

    label_str = processor.batch_decode(
        pred.label_ids,
        group_tokens=False
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {"wer": wer}

In [34]:
training_args = TrainingArguments(
    output_dir="./wav2vec2_librispeech",

    overwrite_output_dir=True,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=2,

    learning_rate=1e-4,

    warmup_steps=100,

    num_train_epochs=3,

    logging_steps=10,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2,

    report_to="none"
)

In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [37]:
print(trainer.evaluate())

/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 69.35991668701172, 'eval_wer': 0.030039811798769453, 'eval_runtime': 14.2693, 'eval_samples_per_second': 18.361, 'eval_steps_per_second': 4.625}


In [38]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Wer
1,629.268800,18.350826,0.047412
2,583.343800,15.183524,0.046688
3,560.188200,13.916338,0.041802


/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

TrainOutput(global_step=1071, training_loss=607.3066571232986, metrics={'train_runtime': 1452.3434, 'train_samples_per_second': 5.895, 'train_steps_per_second': 0.737, 'total_flos': 1.199059118145573e+18, 'train_loss': 607.3066571232986, 'epoch': 3.0})

In [39]:
metrics = trainer.evaluate()
print(metrics)

/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 13.916337966918945, 'eval_wer': 0.041802388707926165, 'eval_runtime': 14.3684, 'eval_samples_per_second': 18.234, 'eval_steps_per_second': 4.593, 'epoch': 3.0}


In [40]:
sample = test_dataset[0]

input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_values).logits

pred_ids = torch.argmax(logits, dim=-1)

prediction = processor.batch_decode(pred_ids)[0]

reference = processor.decode(sample["labels"], group_tokens=False)

print("Prediction:")
print(prediction)

print("\nReference:")
print(reference)

Prediction:
CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS

Reference:
CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS


In [1]:
trainer.save_model("./wav2vec2_librispeech_final")
processor.save_pretrained("./wav2vec2_librispeech_final")

NameError: name 'trainer' is not defined